# 🚗⚡ Three Ways to Give XGBoost a Head Start — using the EV recipe

Hello again, detectives! 🕵️

In our first notebook, **[The EV Buyers Mystery](https://www.kaggle.com/code/cdeotte/fable-5-1-eda-original-data-insights)**, we played dice detective on the original 10,000 people and found the **secret recipe** that decides who buys an electric vehicle:

> **buy score** = 1.2 × (income ÷ 100,000) + 0.6 × (concern for the planet) + 2 × (subsidy) − 1 × (medium range worry) − 3 × (high range worry) + *a little random wobble*, and a person buys if the score is above **5.5**.

Finding a formula is fun. **Using** it is even more fun. In this notebook we build three XGBoost models that all see the same simple features, and we hand them the recipe in three different ways:

| model | how it meets the recipe |
|---|---|
| 🟦 **Model 1 — Baseline** | never hears about it; must discover everything from the raw columns |
| 🟩 **Model 2 — Recipe as a head start** | starts from the recipe's answer and only learns the *corrections* (XGBoost calls this a **base margin**) |
| 🟧 **Model 3 — Recipe as a clue** | gets the buy score as one extra column, like a hint on a sticky note |

Then we compare them fold by fold, average all three, and write a submission file. Let's go!

## Chapter 0 — Tools and data 🧰

In [ ]:
import glob, time, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
from scipy.stats import norm
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
warnings.filterwarnings("ignore"); sns.set_theme(style="whitegrid", palette="Set2", font_scale=1.05)

def find_file(name):
    for root in ["/kaggle/input", ".", ".."]:
        hits = glob.glob(f"{root}/**/{name}", recursive=True)
        if hits: return hits[0]
    raise FileNotFoundError(name)
train = pd.read_csv(find_file("train.csv")); test = pd.read_csv(find_file("test.csv")); sub = pd.read_csv(find_file("sample_submission.csv"))
y = (train.Will_Buy_EV == "Yes").astype(int).values
print(f"train {train.shape[0]:,} rows, test {test.shape[0]:,} rows, buyers {y.mean():.1%}")
try:
    import cupy; DEVICE = "cuda"; print("GPU found - training will be quick")
except Exception:
    DEVICE = "cpu"; print("no GPU - training takes a few minutes on the CPU")

## Chapter 1 — A few simple features 🔧

XGBoost is great at splitting single columns, but it has to work hard to *combine* columns. We can help it with a handful of plain, honest features (no tricks, no magic):

* **worry_score** — the range-anxiety recipe from the EDA notebook: `commute − 5 × chargers at home − 5 × chargers at work − 150 × home charging`. It sums up "how scary is charging for this person" in one number.
* **chargers_total** — chargers near home plus chargers near work.
* **income_x_subsidy** — income (in $100k) times 1 if there is a subsidy. Rich *and* subsidised is a strong combination.
* **concern_x_subsidy** — planet concern times subsidy, for the same reason.

The words (city, car type, ...) are turned into category codes so the trees can use them.

In [ ]:
def make_features(df):
    X = df.drop(columns=[c for c in ["id", "Will_Buy_EV"] if c in df.columns]).copy()
    home = (df.Home_Charging_Possible == "Yes").astype(int); subsidy = (df.Subsidy_Available == "Yes").astype(int)
    X["worry_score"] = df.Daily_Commute_km - 5 * df.Charging_Stations_Near_Home - 5 * df.Charging_Stations_Near_Work - 150 * home
    X["chargers_total"] = df.Charging_Stations_Near_Home + df.Charging_Stations_Near_Work
    X["income_x_subsidy"] = df.Annual_Income_USD / 1e5 * subsidy
    X["concern_x_subsidy"] = df.Environmental_Concern_Level * subsidy
    return X

X_all = make_features(pd.concat([train.drop(columns=["Will_Buy_EV"]), test], ignore_index=True))
for c in X_all.select_dtypes("object").columns: X_all[c] = X_all[c].astype("category").cat.codes    # same codes for train and test
X, X_test = X_all.iloc[:len(train)].reset_index(drop=True), X_all.iloc[len(train):].reset_index(drop=True)
print(f"{X.shape[1]} features:", list(X.columns))

## Chapter 2 — Turning the recipe into a head start 🧮

The recipe gives a **score**; a person buys if `score + wobble > 5.5`. Because the wobble is a bell curve with spread 1, the chance of buying is simply `Φ(score − 5.5)` (Φ is the bell-curve "probability below" function).

XGBoost thinks in **log-odds** (the logit): `log(p / (1 − p))`. So to hand the model our head start we convert the recipe's probability into a logit. That number is the **base margin**: the model's starting guess for every row, before a single tree is grown. The trees then only have to learn what the recipe got wrong.

In [ ]:
def recipe_score(df):
    return (1.2 * df.Annual_Income_USD / 1e5 + 0.6 * df.Environmental_Concern_Level + 2.0 * (df.Subsidy_Available == "Yes")
            - 1.0 * (df.Range_Anxiety_Level == "Medium") - 3.0 * (df.Range_Anxiety_Level == "High")).values
def recipe_logit(df):
    p = np.clip(norm.cdf(recipe_score(df) - 5.5), 1e-6, 1 - 1e-6); return np.log(p / (1 - p))

score_train, score_test = recipe_score(train), recipe_score(test)
margin_train, margin_test = recipe_logit(train), recipe_logit(test)
print(f"recipe alone, AUC on train: {roc_auc_score(y, score_train):.4f}  (that is the head start before any tree is grown)")
fig, ax = plt.subplots(figsize=(9, 3.8)); ax.hist(margin_train, bins=80, color="#8da0cb", edgecolor="white")
ax.set_title("The head start: recipe log-odds for every training person", fontweight="bold"); ax.set_xlabel("base margin (log-odds)"); ax.set_ylabel("people"); plt.show()

## Chapter 3 — One honest training routine for all three models 🏋️

We use **5-fold cross-validation** with a fixed seed: split the training rows into 5 groups, train on 4, predict the 5th, repeat. Every row gets a prediction from a model that never saw it, so the score is honest. Test predictions are the average of the 5 fold models. The only thing that changes between our three models is *how* the recipe is handed over.

In [ ]:
PARAMS = dict(n_estimators=3000, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, tree_method="hist", device=DEVICE, eval_metric="auc", early_stopping_rounds=100)
folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(X, y))

def run_xgb(name, X, X_test, margin=None, margin_test=None):
    oof = np.zeros(len(y)); pred = np.zeros(len(X_test)); imp = np.zeros(X.shape[1]); rounds = []; t0 = time.time()
    for f, (tr_idx, va_idx) in enumerate(folds):
        model = xgb.XGBClassifier(**PARAMS)
        if margin is None:
            model.fit(X.iloc[tr_idx], y[tr_idx], eval_set=[(X.iloc[va_idx], y[va_idx])], verbose=False)
            oof[va_idx] = model.predict_proba(X.iloc[va_idx])[:, 1]; pred += model.predict_proba(X_test)[:, 1] / 5
        else:
            model.fit(X.iloc[tr_idx], y[tr_idx], eval_set=[(X.iloc[va_idx], y[va_idx])], base_margin=margin[tr_idx], base_margin_eval_set=[margin[va_idx]], verbose=False)
            oof[va_idx] = model.predict_proba(X.iloc[va_idx], base_margin=margin[va_idx])[:, 1]; pred += model.predict_proba(X_test, base_margin=margin_test)[:, 1] / 5
        imp += model.feature_importances_ / 5; rounds.append(int(model.best_iteration))
    fold_aucs = [roc_auc_score(y[va], oof[va]) for _, va in folds]
    print(f"{name}: CV AUC = {roc_auc_score(y, oof):.5f} | folds {np.round(fold_aucs, 5).tolist()} | boosting rounds {rounds} | {time.time()-t0:.0f}s")
    return dict(name=name, oof=oof, pred=pred, imp=pd.Series(imp, index=X.columns), fold_aucs=fold_aucs, rounds=rounds)

def plot_importance(res, color):
    imp = res["imp"].sort_values(); fig, ax = plt.subplots(figsize=(9, 5.5)); imp.plot(kind="barh", ax=ax, color=color, edgecolor="white")
    ax.set_title(f"{res['name']} - which columns does it use most?", fontweight="bold"); ax.set_xlabel("importance"); plt.show()

## Chapter 4 — 🟦 Model 1: the baseline

No recipe anywhere. The model must find the buying rule on its own from the raw columns and our four helper features.

In [ ]:
m1 = run_xgb("Model 1 - baseline", X, X_test); plot_importance(m1, "#8da0cb")

It works out on its own that income, subsidy, concern and range worry are the ingredients — exactly the recipe's ingredients. Smart trees! 🌳

## Chapter 5 — 🟩 Model 2: the recipe as a head start (base margin)

Same features, but every row starts from the recipe's log-odds. The trees only learn the *corrections*.

In [ ]:
m2 = run_xgb("Model 2 - recipe as base margin", X, X_test, margin=margin_train, margin_test=margin_test); plot_importance(m2, "#66c2a5")

Two things to notice: the score is a touch higher, and it usually needs **fewer boosting rounds** — it does not have to rediscover what the recipe already told it. The importance chart also changes: the recipe's own ingredients matter a little less, because their main effect is already inside the head start.

## Chapter 6 — 🟧 Model 3: the recipe as a clue (extra feature)

Same features plus one more column: the buy score itself.

In [ ]:
X3, X3_test = X.copy(), X_test.copy(); X3["recipe_score"] = score_train; X3_test["recipe_score"] = score_test
m3 = run_xgb("Model 3 - recipe as a feature", X3, X3_test); plot_importance(m3, "#fc8d62")

The new column goes straight to the top of the importance chart: when you hand a tree a column that already contains the answer's main part, it uses it a lot.

## Chapter 7 — The comparison 🥇🥈🥉

In [ ]:
results = [m1, m2, m3]
summary = pd.DataFrame({"model": [r["name"] for r in results], "CV AUC": [roc_auc_score(y, r["oof"]) for r in results], "mean boosting rounds": [np.mean(r["rounds"]) for r in results]})
for f in range(5): summary[f"fold {f}"] = [r["fold_aucs"][f] for r in results]
summary.set_index("model").style.format({"CV AUC": "{:.5f}", "mean boosting rounds": "{:.0f}", **{f"fold {f}": "{:.5f}" for f in range(5)}}).background_gradient(cmap="Greens", subset=["CV AUC"] + [f"fold {f}" for f in range(5)])

In [ ]:
colors = ["#8da0cb", "#66c2a5", "#fc8d62"]
fig, axes = plt.subplots(1, 2, figsize=(16, 4.8))
w = 0.26; xs = np.arange(5)
for i, r in enumerate(results): axes[0].bar(xs + (i - 1) * w, r["fold_aucs"], width=w, color=colors[i], edgecolor="white", label=r["name"])
lo = min(min(r["fold_aucs"]) for r in results) - 0.0005; axes[0].set_ylim(lo, max(max(r["fold_aucs"]) for r in results) + 0.0005)
axes[0].set_xticks(xs); axes[0].set_xticklabels([f"fold {f}" for f in range(5)]); axes[0].set_ylabel("AUC"); axes[0].set_title("AUC per fold (zoomed in - the differences are small!)", fontweight="bold"); axes[0].legend(fontsize=9)
for i, r in enumerate(results): axes[1].bar(xs + (i - 1) * w, r["rounds"], width=w, color=colors[i], edgecolor="white", label=r["name"])
axes[1].set_xticks(xs); axes[1].set_xticklabels([f"fold {f}" for f in range(5)]); axes[1].set_ylabel("boosting rounds until early stopping"); axes[1].set_title("How long did each model need to learn?", fontweight="bold"); axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

**What we see.** All three models land within a hair of each other — the trees are good enough to find the recipe on their own from 669,000 examples. Handing over the recipe helps a little (Models 2 and 3 are ahead on most folds), and the head-start model gets there with fewer boosting rounds. Notice the left chart is *zoomed in*: the whole difference between the models is a few ten-thousandths of AUC. In a competition, that is exactly the scale where the leaderboard is decided!

## Chapter 8 — Team up: the blend 🤝

The three models agree most of the time, but not always. When several decent models disagree a little, their **average** is usually better than any one of them. Let's check that on our honest out-of-fold predictions and write the submission.

In [ ]:
oof_blend = (m1["oof"] + m2["oof"] + m3["oof"]) / 3; pred_blend = (m1["pred"] + m2["pred"] + m3["pred"]) / 3
print("correlation between the models' predictions:")
print(pd.DataFrame(np.corrcoef([m1["oof"], m2["oof"], m3["oof"]]), index=["M1", "M2", "M3"], columns=["M1", "M2", "M3"]).round(4).to_string())
print(f"\nequal-weight blend of the three: CV AUC = {roc_auc_score(y, oof_blend):.5f}")
scores = pd.Series({r["name"]: roc_auc_score(y, r["oof"]) for r in results} | {"Blend of all three": roc_auc_score(y, oof_blend)})
fig, ax = plt.subplots(figsize=(9, 4)); bars = ax.barh(scores.index, scores.values, color=colors + ["#e78ac3"], edgecolor="white")
ax.set_xlim(scores.min() - 0.0003, scores.max() + 0.0003); ax.set_xlabel("CV AUC (zoomed in)"); ax.set_title("The blend beats every single model", fontweight="bold")
for b, v in zip(bars, scores.values): ax.text(v, b.get_y() + b.get_height() / 2, f" {v:.5f}", va="center", fontweight="bold")
plt.show()
sub["Will_Buy_EV"] = pred_blend; sub.to_csv("submission.csv", index=False); sub.head()

## The end 🏁

What we learned today:

* A formula you discovered in EDA can enter a model in more than one way: as a **base margin** (a head start the trees correct), or as a **feature** (a clue the trees can split on). Both are honest and leak-free, because the recipe only uses each row's own columns.
* On this data the trees are strong enough to rediscover the recipe, so the gains are small — but real, consistent across folds, and the head-start version trains faster.
* **Blending** three slightly different models gave a bigger jump than any single trick. Diversity is the friend of every Kaggler.

Ideas to try next: tune the tree settings with the same 5 folds, add other model families to the blend (LightGBM, CatBoost, a small neural network), and go back to the EDA — the data has more stories to tell.

Happy modelling! 🚗⚡